# C2 — ResNet18 với quy tắc chọn checkpoint v6

Thí nghiệm cuối cùng, và nó vá một lỗ hổng thật: **một nửa ensemble đang khóa
vẫn dùng quy tắc chọn checkpoint cũ**.

| Thành viên | Chọn checkpoint bằng |
|---|---|
| ResNet18 B1 (v4) | **validation AUC** |
| DenseNet121 v5 | spec@sens97 → NLL |

Validation AUC sau đó được chứng minh là bão hòa (biên độ 0,0055 qua mọi epoch,
so với 0,0492 của specificity) và chuyển kém. Nên câu hỏi rất cụ thể:

> ResNet18 có đang giữ nhầm epoch không?

## Đây là ablation về chọn checkpoint, không phải công thức mới

Giữ **nguyên vẹn** từ B1: Adam, lr 1e-4, weight decay 1e-5, 15 epoch,
patience 5, batch 32, ReduceLROnPlateau theo **AUC**, stretch, augment mạnh,
weighted CE, đúng năm fold và seed cũ.

Scheduler vẫn bám AUC là **cố ý** — nó thuộc công thức đang giữ cố định, không
thuộc quy tắc đang thử.

## Thay đổi duy nhất

```
validation AUC
→
spec@sens97 → hòa 0,005 → HSAS@97 → hòa 0,002 → NLL → epoch sớm hơn
```

Mỗi epoch lưu lại prediction mức group, nên sau này kiểm toán được quy tắc và
so sánh với lựa chọn theo AUC mà **không cần train lại**.

## Ba ensemble đăng ký trước

```
E0 = 0,50·ResNet_v4 + 0,50·DenseNet_v5     (đang khóa, 40 FP)
E1 = 0,50·ResNet_v6 + 0,50·DenseNet_v5
E2 = 0,25·ResNet_v4 + 0,25·ResNet_v6 + 0,50·DenseNet_v5
```

E2 giữ tổng trọng số họ ResNet ở 0,5 để hai ResNet không chiếm hai phần ba
ensemble. Không phải trọng số tinh chỉnh trên benchmark.

**E0 nằm trong tập ứng viên.** Nếu nó thắng trên OOF thì benchmark **không được
mở** và mô hình cũ được đóng băng.

## Cấu hình

In [1]:
RUN_MODE      = "auto"   # auto | smoke | full
DATA_ROOT_OVERRIDE = None # ví dụ: "/Users/me/data/chest_xray"
SEED          = 42
IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 15
LR            = 1e-4
WEIGHT_DECAY  = 1e-5
PATIENCE      = 5
NUM_WORKERS   = 2         # runtime sẽ ép về 0 trên macOS

N_FOLDS       = 5         # 1 = một holdout 15%; 5 = cross-validation đầy đủ
VAL_FRACTION  = 0.15      # chỉ dùng khi N_FOLDS = 1
BORDER_FRAC   = 0.15      # dùng ở phần 4.3
DETERMINISTIC = True      # cudnn tất định; chậm hơn một chút, đổi lại tái lập tốt hơn
RESIZE_MODE   = "letterbox"  # mặc định cho thí nghiệm không ghi rõ "resize"
THRESHOLD_OBJECTIVE = "sensitivity"  # sensitivity | balanced_accuracy
TARGET_SENSITIVITY = 0.97
BOOTSTRAP_REPS = 2000    # KTC cho audit tỉ lệ khung ở mức filename-derived group

# Hai bộ augmentation. "mạnh" mô phỏng thiết lập của các cài đặt công khai
# đạt độ đặc hiệu cao hơn: xoay 30 độ, zoom và dịch ảnh.
AUG_PRESETS = {
    "nhe":  {"rotation": 10, "scale": 0.00, "translate": 0.00, "jitter": 0.15},
    "manh": {"rotation": 30, "scale": 0.20, "translate": 0.10, "jitter": 0.20},
}

# C2 tái hiện đúng công thức B1 (stretch_manh của v4). Thay đổi phương pháp
# DUY NHẤT là quy tắc chọn checkpoint; mọi siêu tham số khác giữ nguyên.
ALL_EXPERIMENTS = [
    {"name": "resnet18_v6", "arch": "resnet18", "size": 224, "aug": "manh",
     "balancing": "weighted", "resize": "stretch",
     "hoi": "B1 với quy tắc chọn checkpoint v6"},
]
SMOKE_EXPERIMENTS = [dict(ALL_EXPERIMENTS[0], name="smoke_resnet_v6")]

# Biên hòa của quy tắc v6, khóa trước khi chạy.
SPECIFICITY_TIE = 0.005
HSAS_TIE        = 0.002

# Công thức B1 phải khớp, nếu không thì đây không còn là ablation về checkpoint.
EXPECTED_B1 = {"arch": "resnet18", "size": 224, "resize": "stretch",
               "aug": "manh", "balancing": "weighted"}

EXPERIMENTS = ALL_EXPERIMENTS
CLASSES = ("NORMAL", "PNEUMONIA")  # NORMAL=0, PNEUMONIA=1

In [2]:
import gc, hashlib, json, os, platform, random, re, time, warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import scipy
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

warnings.filterwarnings("ignore", category=UserWarning)

IS_KAGGLE = Path("/kaggle/working").is_dir()
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

if RUN_MODE == "auto":
    RUN_MODE = "full" if IS_KAGGLE and DEVICE.type == "cuda" else "smoke"
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE phải là 'auto', 'smoke' hoặc 'full'")

if RUN_MODE == "smoke":
    # N_FOLDS giữ nguyên để cách chia khớp B1.
    EPOCHS, FOLDS_TO_RUN = 1, [0]
    EXPERIMENTS = SMOKE_EXPERIMENTS
else:
    EXPERIMENTS = ALL_EXPERIMENTS
    FOLDS_TO_RUN = list(range(N_FOLDS))
    _actual = {k: EXPERIMENTS[0][k] for k in EXPECTED_B1}
    if _actual != EXPECTED_B1:
        raise AssertionError(
            f"Công thức lệch B1: {_actual}; kỳ vọng {EXPECTED_B1}")


if platform.system() == "Darwin":
    NUM_WORKERS = 0  # notebook + spawn không an toàn với closure worker/cache global
LOCAL_PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                           if (p / ".git").is_dir()), Path.cwd())
WORK_DIR = (Path("/kaggle/working") if IS_KAGGLE
            else LOCAL_PROJECT_ROOT / "artifacts/notebook_rerun")
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = WORK_DIR / "train_log_c2.txt"
LOG_PATH.write_text("", encoding="utf-8")
PIN_MEMORY = DEVICE.type == "cuda"
AMP_ENABLED = DEVICE.type == "cuda"
DEVICE_NAME = (torch.cuda.get_device_name(0) if DEVICE.type == "cuda"
               else "Apple Metal (MPS)" if DEVICE.type == "mps" else platform.processor() or "CPU")

if DETERMINISTIC and DEVICE.type == "cuda":
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def set_seed(seed=SEED):
    """Seed mọi nguồn ngẫu nhiên mà pipeline đụng tới."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def loader_seed_args(seed=SEED):
    """generator + worker_init_fn cho DataLoader.

    Thiếu hai thứ này thì thứ tự xáo trộn và augmentation chạy trong worker vẫn
    ngẫu nhiên dù đã gọi set_seed — một lỗ hổng tái lập rất hay bị bỏ sót.
    """
    generator = torch.Generator()
    generator.manual_seed(seed)

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)

    return {"generator": generator, "worker_init_fn": worker_init_fn}


def log(*parts):
    """In ra màn hình, đồng thời ghi vào LOG_PATH.

    Output của notebook Kaggle có thể mất chunk khi in nhanh; file thì không.
    """
    line = " ".join(str(part) for part in parts)
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")


set_seed()

# Ghi lại phiên bản thư viện. Thiếu nó thì con số trong báo cáo không gắn được
# với môi trường đã sinh ra chúng.
VERSIONS = {
    "python": platform.python_version(), "torch": torch.__version__,
    "torchvision": torchvision.__version__, "numpy": np.__version__,
    "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "pillow": PIL.__version__,
}
with open(WORK_DIR / "environment.json", "w") as handle:
    json.dump({**VERSIONS, "device": str(DEVICE), "device_name": DEVICE_NAME,
               "run_mode": RUN_MODE, "seed": SEED,
               "deterministic": DETERMINISTIC}, handle, indent=2)

# Lưu cấu hình đã resolve sau khi auto/smoke/full được áp dụng. File này giúp
# phân biệt source config với config thực sự sinh ra kết quả.
RESOLVED_CONFIG = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "image_cache_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "n_folds": N_FOLDS,
    "val_fraction": VAL_FRACTION,
    "deterministic": DETERMINISTIC,
    "threshold_objective": THRESHOLD_OBJECTIVE,
    "target_sensitivity": TARGET_SENSITIVITY,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "augment_presets": AUG_PRESETS,
    "experiments": EXPERIMENTS,
}
with open(WORK_DIR / "resolved_config.json", "w", encoding="utf-8") as handle:
    json.dump(RESOLVED_CONFIG, handle, indent=2, ensure_ascii=False)

log("runtime:", "Kaggle" if IS_KAGGLE else "local", "| mode:", RUN_MODE)
log("device:", DEVICE, f"({DEVICE_NAME})", "| AMP:", AMP_ENABLED,
    "| workers:", NUM_WORKERS)
if RUN_MODE == "smoke":
    log("SMOKE RUN: chỉ kiểm tra pipeline; KHÔNG dùng chỉ số để báo cáo.")
log("phiên bản:", " ".join(f"{k}={v}" for k, v in VERSIONS.items()))

runtime: Kaggle | mode: full
device: cuda (Tesla T4) | AMP: True | workers: 2
phiên bản: python=3.12.13 torch=2.10.0+cu128 torchvision=0.25.0+cu128 numpy=2.0.2 pandas=2.3.3 scikit-learn=1.6.1 scipy=1.16.3 pillow=11.3.0


# 1. Dữ liệu

In [3]:
def list_images(directory):
    """Ảnh .jpeg thật, bỏ file sidecar ._* của macOS."""
    return sorted(p for p in Path(directory).glob("*.jpeg")
                  if not p.name.startswith("._"))


def find_data_root(search_paths):
    """Thư mục chứa trực tiếp train/NORMAL và train/PNEUMONIA."""
    if isinstance(search_paths, (str, Path)):
        search_paths = [search_paths]

    candidates = []
    for base in map(Path, search_paths):
        if not base.exists():
            continue
        for train_dir in base.rglob("train"):
            if "__MACOSX" in train_dir.parts:
                continue
            if (train_dir / "NORMAL").is_dir() and (train_dir / "PNEUMONIA").is_dir():
                candidates.append(train_dir.parent.resolve())

    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy dataset dưới {[str(p) for p in search_paths]}. "
            "Kaggle: Add Data 'Chest X-Ray Images (Pneumonia)'. "
            "Mac: đặt DATA_ROOT_OVERRIDE hoặc CXR_DATA_ROOT.")

    candidates = sorted(set(candidates), key=lambda path: len(path.parts))
    for candidate in candidates:
        n = len(list_images(candidate / "train" / "NORMAL"))
        mark = "  <- dùng" if candidate == candidates[0] else "  (bản trùng, bỏ qua)"
        print(f"  {candidate}  [{n} ảnh train/NORMAL]{mark}")
    return candidates[0]


explicit_root = DATA_ROOT_OVERRIDE or os.environ.get("CXR_DATA_ROOT")
if explicit_root:
    DATA_ROOT = find_data_root([explicit_root])
else:
    cwd = Path.cwd()
    DATA_ROOT = find_data_root([
        "/kaggle/input", cwd / "chest_xray", cwd.parent / "chest_xray",
        cwd.parent.parent / "chest_xray", cwd / "data/raw",
        cwd.parent / "data/raw",
    ])
log("\nDATA_ROOT =", DATA_ROOT)

  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray  [1341 ảnh train/NORMAL]  <- dùng
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray  [1341 ảnh train/NORMAL]  (bản trùng, bỏ qua)

DATA_ROOT = /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray


In [4]:
PNEUMONIA_RE = re.compile(r"^person(\d+)_(bacteria|virus)_", re.IGNORECASE)
NORMAL_RE    = re.compile(r"^(?:(NORMAL\d+)-)?IM-(\d+)-", re.IGNORECASE)


def parse_group_id(filename):
    """Khoá group suy từ tên file; không khẳng định đây là clinical patient ID."""
    match = PNEUMONIA_RE.match(filename)
    if match:
        return f"pneumonia:{match.group(2).lower()}:{int(match.group(1))}"
    match = NORMAL_RE.match(filename)
    if match:
        return f"normal:{(match.group(1) or 'IM').lower()}:{int(match.group(2))}"
    raise ValueError(f"Tên file lạ, không suy ra được group: {filename}")


def build_manifest(root):
    """Một dòng cho mỗi ảnh: đường dẫn, split gốc, nhãn, filename-derived group."""
    rows = []
    for split in ("train", "val", "test"):
        for class_id, class_name in enumerate(CLASSES):
            directory = Path(root) / split / class_name
            if not directory.is_dir():
                continue
            for path in list_images(directory):
                rows.append({
                    "path": str(path.resolve()), "filename": path.name,
                    "split_original": split, "class_name": class_name,
                    "class_id": class_id, "group_id": parse_group_id(path.name)})
    if not rows:
        raise FileNotFoundError(f"Không có ảnh .jpeg nào dưới {root}")

    frame = pd.DataFrame(rows)
    frame["cache_index"] = np.arange(len(frame))   # vị trí trong cache ở mục 2.2
    return frame


manifest = build_manifest(DATA_ROOT)
log(f"{len(manifest):,} ảnh | {manifest['group_id'].nunique():,} filename-derived groups")
manifest.head(3)

5,856 ảnh | 4,097 filename-derived groups


,path,filename,split_original,class_name,class_id,group_id,cache_index
0,/kaggle/input/datasets/paultimothymooney/chest...,IM-0115-0001.jpeg,train,NORMAL,0,normal:im:115,0
1,/kaggle/input/datasets/paultimothymooney/chest...,IM-0117-0001.jpeg,train,NORMAL,0,normal:im:117,1
2,/kaggle/input/datasets/paultimothymooney/chest...,IM-0119-0001.jpeg,train,NORMAL,0,normal:im:119,2


In [5]:
print("1.3.1  Số lượng theo split và lớp")
print("-" * 62)
print(f"{'split':<8}{'NORMAL':>9}{'PNEUMONIA':>12}{'tổng':>9}{'P/N':>7}")
for split in ("train", "val", "test"):
    subset = manifest[manifest["split_original"] == split]
    counts = subset["class_name"].value_counts()
    normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
    ratio = pneumonia / normal if normal else float("nan")
    print(f"{split:<8}{normal:>9,}{pneumonia:>12,}{normal + pneumonia:>9,}{ratio:>7.2f}")
print(f"{'TỔNG':<8}{'':>9}{'':>12}{len(manifest):>9,}")

1.3.1  Số lượng theo split và lớp
--------------------------------------------------------------
split      NORMAL   PNEUMONIA     tổng    P/N
train       1,341       3,875    5,216   2.89
val             8           8       16   1.00
test          234         390      624   1.67
TỔNG                             5,856


In [6]:
print("1.3.2  Ảnh trùng nội dung (SHA-256)")
print("-" * 62)
manifest["sha256"] = [hashlib.sha256(Path(path).read_bytes()).hexdigest()
                      for path in manifest["path"]]
by_hash = defaultdict(list)
for digest, split in zip(manifest["sha256"], manifest["split_original"]):
    by_hash[digest].append(split)

duplicates = [s for s in by_hash.values() if len(s) > 1]
cross_split = [s for s in duplicates if len(set(s)) > 1]
print(f"tổng file             : {len(manifest):,}")
print(f"hash duy nhất         : {len(by_hash):,}")
print(f"nhóm ảnh trùng        : {len(duplicates)}")
print(f"  trong đó xuyên split: {len(cross_split)}")
print()
print("Không có ảnh y hệt nằm xuyên original split." if not cross_split
      else "CẢNH BÁO: ảnh trùng xuyên original split — kết quả đánh giá bị nhiễm.")

1.3.2  Ảnh trùng nội dung (SHA-256)
--------------------------------------------------------------
tổng file             : 5,856
hash duy nhất         : 5,824
nhóm ảnh trùng        : 30
  trong đó xuyên split: 0

Không có ảnh y hệt nằm xuyên original split.


In [7]:
print("1.3.4  Filename-derived group và nguy cơ trùng giữa các split")
print("-" * 62)
naive_re = re.compile(r"^(person\d+)_", re.IGNORECASE)
naive, corrected = defaultdict(set), defaultdict(set)
for filename, split in zip(manifest["filename"], manifest["split_original"]):
    match = naive_re.match(filename)
    naive[match.group(1).lower() if match else filename].add(split)
    corrected[parse_group_id(filename)].add(split)

naive_span     = sum(1 for s in naive.values() if len(s) > 1)
corrected_span = sum(1 for s in corrected.values() if len(s) > 1)
print(f"khoá person<N>              : {len(naive):,} nhóm, {naive_span} nằm ở >1 split")
print(f"khoá (phân nhóm, person<N>) : {len(corrected):,} nhóm, {corrected_span} nằm ở >1 split")

subtype_ids = defaultdict(set)
for filename in manifest["filename"]:
    match = PNEUMONIA_RE.match(filename)
    if match:
        subtype_ids[match.group(2).lower()].add(int(match.group(1)))
print()
for subtype, ids in sorted(subtype_ids.items()):
    print(f"  {subtype:<9}: {len(ids):,} số, dải 1..{max(ids)}, "
          f"mật độ {len(ids) / max(ids):.3f}")
print(f"  số được dùng bởi CẢ HAI phân nhóm: "
      f"{len(subtype_ids['bacteria'] & subtype_ids['virus']):,}")

per_group = Counter(parse_group_id(f) for f in manifest["filename"])
multi = sum(1 for n in per_group.values() if n > 1)
print(f"\nnhóm có >1 ảnh: {multi:,}/{len(per_group):,} "
      f"(nhiều nhất {max(per_group.values())} ảnh)")

1.3.4  Filename-derived group và nguy cơ trùng giữa các split
--------------------------------------------------------------
khoá person<N>              : 3,257 nhóm, 170 nằm ở >1 split
khoá (phân nhóm, person<N>) : 4,097 nhóm, 0 nằm ở >1 split

  bacteria : 1,437 số, dải 1..1954, mật độ 0.735
  virus    : 1,216 số, dải 1..1685, mật độ 0.722
  số được dùng bởi CẢ HAI phân nhóm: 979

nhóm có >1 ảnh: 726/4,097 (nhiều nhất 30 ảnh)


# 2. Phương pháp

In [8]:
def label_split(manifest, train, val, test):
    out = manifest.copy()
    out["split"] = pd.NA
    out.loc[train.index, "split"] = "train"
    out.loc[val.index,   "split"] = "val"
    out.loc[test.index,  "split"] = "test"
    return out


def make_folds(manifest, n_folds=N_FOLDS, val_fraction=VAL_FRACTION, seed=SEED):
    """Danh sách manifest, mỗi phần tử là một fold đã gán cột split."""
    pool = manifest[manifest["split_original"].isin(["train", "val"])]
    test = manifest[manifest["split_original"] == "test"]
    n_splits = max(2, round(1 / val_fraction)) if n_folds == 1 else n_folds
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    folds = [label_split(manifest, pool.iloc[train_idx], pool.iloc[val_idx], test)
             for train_idx, val_idx in
             splitter.split(pool, pool["class_id"], groups=pool["group_id"])]
    return folds[:1] if n_folds == 1 else folds


def count_leaked_groups(split):
    """Số filename-derived groups xuất hiện ở nhiều split. Phải bằng 0."""
    return int((split.groupby("group_id")["split"].nunique() > 1).sum())


def count_leaked_hashes(split):
    """Số nội dung ảnh y hệt xuất hiện ở nhiều split. Phải bằng 0."""
    return int((split.groupby("sha256")["split"].nunique() > 1).sum())


def split_summary(split):
    rows = []
    for name in ("train", "val", "test"):
        subset = split[split["split"] == name]
        counts = subset["class_name"].value_counts()
        normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
        rows.append({"split": name, "NORMAL": normal, "PNEUMONIA": pneumonia,
                     "tổng": normal + pneumonia,
                     "groups": subset["group_id"].nunique(),
                     "P/N": round(pneumonia / max(normal, 1), 2)})
    return pd.DataFrame(rows).set_index("split")


FOLDS = make_folds(manifest)
log(f"\n{len(FOLDS)} fold, chia theo filename-derived group:")
for i, split in enumerate(FOLDS):
    s = split_summary(split)
    log(f"  fold {i}: train {s.loc['train','tổng']:>5,}  val {s.loc['val','tổng']:>4,}  "
        f"test {s.loc['test','tổng']:>4,}  |  group/hash ở >1 split: "
        f"{count_leaked_groups(split)}/{count_leaked_hashes(split)}")
    assert count_leaked_groups(split) == 0
    assert count_leaked_hashes(split) == 0
    split.to_csv(WORK_DIR / f"manifest_fold{i}.csv", index=False)

print("\nChi tiết fold 0:")
display(split_summary(FOLDS[0]))


5 fold, chia theo filename-derived group:
  fold 0: train 4,168  val 1,064  test  624  |  group/hash ở >1 split: 0/0
  fold 1: train 4,224  val 1,008  test  624  |  group/hash ở >1 split: 0/0
  fold 2: train 4,165  val 1,067  test  624  |  group/hash ở >1 split: 0/0
  fold 3: train 4,190  val 1,042  test  624  |  group/hash ở >1 split: 0/0
  fold 4: train 4,181  val 1,051  test  624  |  group/hash ở >1 split: 0/0

Chi tiết fold 0:


,NORMAL,PNEUMONIA,tổng,groups,P/N
split,,,,,
train,1092,3076,4168,2934,2.82
val,257,807,1064,735,3.14
test,234,390,624,428,1.67


## 2.2. Tiền xử lý và augmentation

In [9]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
MEAN_T = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_T  = torch.tensor(IMAGENET_STD,  device=DEVICE).view(1, 3, 1, 1)


def device_augment(batch, cfg):
    """Lật, xoay, zoom, dịch, đổi sáng/tương phản trên CUDA/MPS/CPU.

    Làm bằng PIL trong DataLoader thì CPU thành nút thắt: riêng RandomRotation
    và ColorJitter đã ngốn hơn 1.000 lần thời gian đọc cache. Ở đây mọi phép
    biến đổi là tensor op chạy theo lô trên accelerator đang chọn.

    Xoay, zoom và dịch được gộp vào MỘT phép biến đổi affine, nên chỉ nội suy
    một lần thay vì ba lần chồng lên nhau.
    """
    n, dev = batch.size(0), batch.device
    flip = torch.rand(n, device=dev) < 0.5
    batch = torch.where(flip.view(-1, 1, 1, 1), batch.flip(-1), batch)

    rand = lambda: torch.rand(n, device=dev) * 2 - 1          # noqa: E731  -1..1
    angles = rand() * (cfg["rotation"] * np.pi / 180)
    zoom = 1 + rand() * cfg["scale"]
    cos, sin = torch.cos(angles) * zoom, torch.sin(angles) * zoom
    theta = torch.zeros(n, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 1] = cos, -sin
    theta[:, 1, 0], theta[:, 1, 1] = sin, cos
    theta[:, 0, 2] = rand() * cfg["translate"]
    theta[:, 1, 2] = rand() * cfg["translate"]
    grid = F.affine_grid(theta, batch.shape, align_corners=False)
    batch = F.grid_sample(batch, grid, align_corners=False, padding_mode="zeros")

    j = cfg["jitter"]
    scale = 1 + rand().view(-1, 1, 1, 1) * j
    contrast = 1 + rand().view(-1, 1, 1, 1) * j
    mean = batch.mean(dim=(1, 2, 3), keepdim=True)
    return ((batch * scale - mean) * contrast + mean).clamp(0, 1)


def to_model_input(batch_uint8, size=IMG_SIZE, aug=None):
    """(B,H,W) uint8 -> (B,3,H,W) chuẩn hoá trên DEVICE.

    Cache giữ ảnh ở IMG_SIZE; thí nghiệm nào cần kích thước khác thì thu nhỏ
    ngay trên DEVICE. Đây là resize hai bước (gốc -> IMG_SIZE -> size), áp dụng
    đồng nhất cho mọi split nên không tạo chênh lệch giữa train và test.
    """
    x = batch_uint8.to(DEVICE, non_blocking=True).float().div_(255).unsqueeze(1)
    if size != IMG_SIZE:
        x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    if aug is not None:
        x = device_augment(x, aug)
    return (x.expand(-1, 3, -1, -1) - MEAN_T) / STD_T


def resize_for_cache(image, size=IMG_SIZE, mode=RESIZE_MODE):
    gray = image.convert("L")
    if mode == "stretch":
        return np.asarray(gray.resize((size, size), Image.Resampling.BILINEAR))
    if mode != "letterbox":
        raise ValueError(f"RESIZE_MODE lạ: {mode}")
    gray.thumbnail((size, size), Image.Resampling.BILINEAR)
    array = np.asarray(gray)
    fill = int(np.median(array))
    canvas = Image.new("L", (size, size), color=fill)
    offset = ((size - gray.width) // 2, (size - gray.height) // 2)
    canvas.paste(gray, offset)
    return np.asarray(canvas)


def build_image_cache(manifest, size=IMG_SIZE, mode=RESIZE_MODE):
    cache = np.zeros((len(manifest), size, size), dtype=np.uint8)
    for position, path in enumerate(manifest["path"]):
        with Image.open(path) as image:
            cache[position] = resize_for_cache(image, size, mode)
        if (position + 1) % 1500 == 0:
            print(f"  {position + 1:,}/{len(manifest):,}")
    return cache


# Mỗi chế độ resize cần một cache riêng. Chỉ dựng những chế độ thực sự được
# dùng, để so sánh stretch với letterbox nằm trong cùng một lần chạy thay vì hai
# lần chạy khác nhau như trước.
REQUIRED_MODES = sorted({spec.get("resize", RESIZE_MODE) for spec in EXPERIMENTS})
IMAGE_CACHES = {}
for mode in REQUIRED_MODES:
    _started = time.time()
    IMAGE_CACHES[mode] = build_image_cache(manifest, mode=mode)
    log(f"cache {mode}: {IMAGE_CACHES[mode].nbytes / 1e6:.0f} MB cho "
        f"{len(manifest):,} ảnh trong {time.time() - _started:.0f}s")

# Cache mặc định cho các đoạn không gắn với một thí nghiệm cụ thể.
IMAGE_CACHE = IMAGE_CACHES[RESIZE_MODE if RESIZE_MODE in IMAGE_CACHES
                           else REQUIRED_MODES[0]]


class XRayDataset(Dataset):
    """Trả về ảnh uint8 thô; augmentation diễn ra trên DEVICE."""

    def __init__(self, rows, mode=RESIZE_MODE):
        self.rows, self.mode = rows, mode

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        cache_index, label = self.rows[index]
        return torch.from_numpy(IMAGE_CACHES[self.mode][cache_index]), label


def make_loader(split, name, batch_size=BATCH_SIZE, seed=SEED, mode=RESIZE_MODE):
    subset = split[split["split"] == name]
    return DataLoader(
        XRayDataset(list(zip(subset["cache_index"], subset["class_id"])), mode),
        batch_size=batch_size, shuffle=(name == "train"),
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
        **loader_seed_args(seed))


def make_loaders(split, batch_size=BATCH_SIZE, seed=SEED, mode=RESIZE_MODE):
    return {name: make_loader(split, name, batch_size, seed, mode)
            for name in ("train", "val")}


def class_weights_from(split):
    """Trọng số nghịch tần suất, chuẩn hoá để loss giữ nguyên thang đo."""
    counts = Counter(split[split["split"] == "train"]["class_id"])
    total = sum(counts.values())
    return torch.tensor([total / (len(CLASSES) * counts[i]) for i in range(len(CLASSES))],
                        dtype=torch.float, device=DEVICE)

  1,500/5,856
  3,000/5,856
  4,500/5,856
cache stretch: 294 MB cho 5,856 ảnh trong 54s


## 2.3. Chỉ số đánh giá

In [10]:
METRIC_COLS = ["accuracy", "precision", "recall", "specificity",
               "f1", "bal_acc", "auc", "pr_auc"]


def metrics_at(labels, probs, threshold=0.5):
    """Chấm điểm tại một ngưỡng. threshold=0.5 chính là argmax trên hai logit."""
    labels, probs = np.asarray(labels), np.asarray(probs)
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sensitivity, specificity = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    return {
        "threshold": float(threshold),
        "accuracy": float((labels == preds).mean()),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "specificity": float(specificity),
        "f1": f1_score(labels, preds, zero_division=0),
        "bal_acc": float((sensitivity + specificity) / 2),
        "auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs),
        "confusion_matrix": confusion_matrix(labels, preds).tolist(),
    }


def to_group_level(group_ids, labels, probs):
    """Gộp theo filename-derived group; xác suất là trung bình các ảnh."""
    frame = pd.DataFrame({"group": group_ids, "label": labels, "prob": probs})
    label_counts = frame.groupby("group")["label"].nunique()
    if int(label_counts.max()) != 1:
        bad = label_counts[label_counts > 1].index.tolist()[:5]
        raise ValueError(f"Group chứa nhiều nhãn, ví dụ: {bad}")
    rolled = frame.groupby("group", sort=True).agg(
        label=("label", "first"), prob=("prob", "mean"))
    return rolled["label"].to_numpy(), rolled["prob"].to_numpy()


def tune_threshold(labels, probs, objective=THRESHOLD_OBJECTIVE,
                   target_sensitivity=TARGET_SENSITIVITY):
    """Chọn một candidate thật trên validation/OOF, không nội suy qua vùng tie."""
    labels, probs = np.asarray(labels), np.asarray(probs)
    candidates = np.unique(np.clip(probs, 0.001, 0.999))
    if len(candidates) > 400:
        indices = np.linspace(0, len(candidates) - 1, 400).round().astype(int)
        candidates = candidates[np.unique(indices)]

    rows = [metrics_at(labels, probs, float(t)) for t in candidates]
    if objective == "balanced_accuracy":
        best = max(rows, key=lambda m: (m["bal_acc"], m["specificity"],
                                        m["recall"], m["threshold"]))
    elif objective == "sensitivity":
        feasible = [m for m in rows if m["recall"] >= target_sensitivity - 1e-12]
        if not feasible:
            warnings.warn("Không có ngưỡng đạt target sensitivity; dùng recall cao nhất.")
            feasible = rows
            best = max(feasible, key=lambda m: (m["recall"], m["specificity"],
                                                m["threshold"]))
        else:
            best = max(feasible, key=lambda m: (m["specificity"], m["bal_acc"],
                                                m["threshold"]))
    else:
        raise ValueError(f"THRESHOLD_OBJECTIVE lạ: {objective}")
    return float(best["threshold"]), best

## 2.4. Kiến trúc

In [11]:
class SmallCNN(nn.Module):
    """4 khối Conv-BN-ReLU-Pool rồi gộp toàn cục. Train từ đầu."""

    def __init__(self, num_classes=len(CLASSES)):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64),
                                      block(64, 128), block(128, 256))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(arch, pretrained=True, device=DEVICE):
    if arch == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    elif arch == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        model = models.densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features, len(CLASSES))
    elif arch == "small_cnn":
        model = SmallCNN()
    else:
        raise ValueError(f"Kiến trúc lạ: {arch!r}")
    return model.to(device)


def target_layer_for(model, arch):
    """Lớp tích chập cuối để gắn Grad-CAM. Chỉ đích danh theo kiến trúc.

    Cách dò 'Conv2d cuối cùng' sẽ sai âm thầm khi đổi kiến trúc — heatmap vẫn
    hiện ra, chỉ là hiện sai chỗ.
    """
    return {"resnet18": lambda: model.layer4[-1],
            "densenet121": lambda: model.features.denseblock4,
            "small_cnn": lambda: model.features[-1]}[arch]()


def parameter_count_millions(arch):
    model = build_model(arch, pretrained=False, device=torch.device("cpu"))
    count = sum(p.numel() for p in model.parameters()) / 1e6
    del model
    return round(count, 1)


display(pd.DataFrame([
    {"thí nghiệm": s["name"], "kiến trúc": s["arch"], "px": s["size"],
     "resize": s.get("resize", RESIZE_MODE),
     "augment": s["aug"], "balancing": s["balancing"],
     "tham số (M)": parameter_count_millions(s["arch"]),
     "câu hỏi": s["hoi"]}
    for s in EXPERIMENTS]).set_index("thí nghiệm"))

,kiến trúc,px,resize,augment,balancing,tham số (M),câu hỏi
thí nghiệm,,,,,,,
resnet18_v6,resnet18,224,stretch,manh,weighted,11.2,B1 với quy tắc chọn checkpoint v6


## 2.5. Hàm dùng chung

In [12]:
@torch.no_grad()
def predict(model, loader, size=IMG_SIZE):
    """(nhãn thật, xác suất PNEUMONIA) trên toàn bộ loader."""
    model.eval()
    labels_all, probs_all = [], []
    for images, labels in loader:
        logits = model(to_model_input(images, size))
        probs_all += torch.softmax(logits.float(), dim=1)[:, 1].cpu().tolist()
        labels_all += labels.tolist()
    return np.array(labels_all), np.array(probs_all)

## 2.6. Quy tắc chọn checkpoint v6

Định nghĩa sau các hàm cũ nên chúng ghi đè bản v4.

In [13]:
def exact_threshold_at_sensitivity(labels, probs, target=TARGET_SENSITIVITY):
    """Highest observed score that still meets a minimum sensitivity.

    Args:
        labels: Binary labels.
        probs: Predicted probabilities.
        target: Minimum sensitivity to hold.

    Returns:
        The selected threshold, or 0.0 when the target is unreachable.
    """
    labels, probs = np.asarray(labels), np.asarray(probs, dtype=float)
    positive = probs[labels == 1]
    if not len(positive):
        return 0.5
    candidates = np.unique(probs)
    feasible = candidates[[(positive >= c).mean() >= target for c in candidates]]
    return float(feasible.max()) if len(feasible) else 0.0


def specificity_at_sensitivity(labels, probs, target=TARGET_SENSITIVITY):
    """Best specificity reachable while holding a minimum sensitivity.

    Args:
        labels: Binary labels.
        probs: Predicted probabilities.
        target: Minimum sensitivity to hold.

    Returns:
        Tuple of (specificity, threshold).
    """
    threshold = exact_threshold_at_sensitivity(labels, probs, target)
    negative = np.asarray(probs, dtype=float)[np.asarray(labels) == 0]
    if not len(negative):
        return 0.0, threshold
    return float((negative < threshold).mean()), threshold


def hsas_97(labels, probs, min_sensitivity=TARGET_SENSITIVITY):
    """Mean specificity held across sensitivities from the target to 1.

    Not partial AUC: the McClish normalisation divides by the region's own
    width, which cancels the quantity that matters. A model paying more false
    positives to reach 97% sensitivity traces a wider region of the same shape
    and would score identically.

    Args:
        labels: Binary labels.
        probs: Predicted probabilities.
        min_sensitivity: Lower bound of the sensitivity range.

    Returns:
        Mean specificity over the range, in [0, 1].

    Raises:
        ValueError: If either class is missing.
    """
    labels = np.asarray(labels)
    if len(np.unique(labels)) < 2:
        raise ValueError(f"HSAS cần cả hai lớp; thấy {np.unique(labels).tolist()}")
    fpr, tpr, _ = roc_curve(labels, np.asarray(probs, dtype=float),
                            drop_intermediate=False)
    order = np.lexsort((fpr, tpr))
    tpr, fpr = tpr[order], fpr[order]
    keep = np.r_[True, np.diff(tpr) > 0]
    tpr, fpr = tpr[keep], fpr[keep]
    grid = np.linspace(min_sensitivity, 1.0, 512)
    return float(np.trapezoid(1.0 - np.interp(grid, tpr, fpr), grid)
                 / (1.0 - min_sensitivity))


def better_checkpoint(candidate, incumbent):
    """The v6 hierarchy, with the reason recorded.

    Args:
        candidate: Metrics for the current epoch.
        incumbent: Metrics for the held checkpoint, or None.

    Returns:
        Tuple of (replace, reason).
    """
    if incumbent is None:
        return True, "first"
    gap = candidate["specificity"] - incumbent["specificity"]
    if gap > SPECIFICITY_TIE:
        return True, "higher_specificity"
    if gap < -SPECIFICITY_TIE:
        return False, "lower_specificity"
    difference = candidate["hsas_97"] - incumbent["hsas_97"]
    if difference > HSAS_TIE:
        return True, "specificity_tie_higher_hsas"
    if difference < -HSAS_TIE:
        return False, "specificity_tie_lower_hsas"
    if candidate["nll"] < incumbent["nll"] - 1e-9:
        return True, "specificity_hsas_tie_lower_nll"
    return False, "all_tied_keep_earlier"


def group_scores(labels, probs, groups):
    """Collapse image predictions to one score per filename-derived group."""
    frame = pd.DataFrame({"g": groups, "y": labels, "p": probs})
    rolled = frame.groupby("g").agg(y=("y", "first"), p=("p", "mean"))
    return rolled["y"].to_numpy(), rolled["p"].to_numpy()


def group_nll(labels, probs):
    """Unweighted log-loss at group level."""
    p = np.clip(probs, 1e-7, 1 - 1e-7)
    return float(-np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p)))


def group_brier(labels, probs):
    """Brier score at group level."""
    return float(np.mean((probs - labels) ** 2))

## 2.7. Vòng huấn luyện

In [14]:
def run_fold(spec, fold_index, epochs=EPOCHS):
    """Train one fold with B1's recipe and the v6 checkpoint rule.

    The optimizer, scheduler, learning rate, epoch budget and patience are
    exactly B1's. In particular the scheduler still follows validation AUC, as
    it did in v4: changing it would make this something other than a
    checkpoint-selection ablation.

    Args:
        spec: Experiment specification.
        fold_index: Which fold to run.
        epochs: Maximum epochs.

    Returns:
        Result mapping in the shape the other runs produced.
    """
    log(f"\n{'=' * 62}\n{spec['name']}  |  fold {fold_index}\n{'=' * 62}")
    resize, size = spec["resize"], spec["size"]
    aug = AUG_PRESETS[spec["aug"]]
    set_seed(SEED + fold_index)

    split = FOLDS[fold_index]
    loaders = make_loaders(split, seed=SEED + fold_index, mode=resize)
    model = build_model(spec["arch"], pretrained=True)

    weights = (class_weights_from(split) if spec["balancing"] == "weighted"
               else None)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR,
                                 weight_decay=WEIGHT_DECAY)
    # Unchanged from B1, including following AUC: the scheduler is part of the
    # recipe being held fixed, not part of the selection rule being tested.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.3, patience=2)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

    val_rows = split[split["split"] == "val"].reset_index(drop=True)
    val_groups = val_rows["group_id"].to_numpy()
    log(f"{spec['arch']} | {size}px | resize {resize} | augment {spec['aug']} | "
        f"balancing {spec['balancing']} | lr {LR:.0e} | epochs {epochs}")
    log("chọn checkpoint: spec@sens97 → HSAS@97 → NLL → epoch sớm hơn")

    best, best_epoch, best_state, stale, history = None, 0, None, 0, []
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for images, labels in loaders["train"]:
            inputs = to_model_input(images, size, aug)
            labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                loss = criterion(model(inputs), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * inputs.size(0)

        labels, probs = predict(model, loaders["val"], size)
        g_labels, g_probs = group_scores(labels, probs, val_groups)
        specificity, threshold = specificity_at_sensitivity(g_labels, g_probs)
        operating = metrics_at(g_labels, g_probs, threshold)
        (tn, fp), (fn, tp) = operating["confusion_matrix"]
        auc = roc_auc_score(g_labels, g_probs)
        scheduler.step(auc)

        current = {
            "epoch": epoch, "train_loss": running / len(loaders["train"].dataset),
            "specificity": specificity, "sensitivity": operating["recall"],
            "hsas_97": hsas_97(g_labels, g_probs), "threshold": threshold,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "nll": group_nll(g_labels, g_probs),
            "brier": group_brier(g_labels, g_probs), "auc": auc,
            "pr_auc": average_precision_score(g_labels, g_probs),
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
        }
        replace, reason = better_checkpoint(current, best)
        current["selected"] = replace
        current["selection_reason"] = reason
        history.append(current)

        # Every epoch's group predictions are kept, so the rule can be audited
        # afterwards -- and an alternative rule compared -- without retraining.
        pd.DataFrame({"group_id": np.unique(val_groups), "label": g_labels,
                      "p_pneumonia": g_probs}).to_csv(
            WORK_DIR / f"epoch_predictions_{spec['name']}_fold{fold_index}"
                       f"_epoch{epoch:02d}.csv", index=False)

        if replace:
            best, best_epoch, stale = current, epoch, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            stale += 1

        log(f"epoch {epoch:>2}/{epochs}  loss {current['train_loss']:.4f}  "
            f"spec@sens{TARGET_SENSITIVITY:.0%} {specificity:.4f}  "
            f"HSAS {current['hsas_97']:.4f}  AUC {auc:.4f}  "
            f"NLL {current['nll']:.4f}  {reason}"
            f"{'  <- best' if replace else ''}")
        if stale >= PATIENCE:
            log(f"dừng sớm ở epoch {epoch}")
            break

    model.load_state_dict(best_state)
    best_auc_epoch = max(history, key=lambda h: h["auc"])["epoch"]
    log(f"giữ checkpoint epoch {best_epoch} ({best['selection_reason']}); "
        f"quy tắc AUC cũ sẽ chọn epoch {best_auc_epoch}"
        f"{' — trùng nhau' if best_epoch == best_auc_epoch else ' — KHÁC'}")

    val_labels, val_probs = predict(model, loaders["val"], size)
    tag = f"{spec['name']}_fold{fold_index}"
    torch.save(best_state, WORK_DIR / f"{tag}.pth")
    val_rows.assign(p_pneumonia=val_probs).to_csv(
        WORK_DIR / f"validation_predictions_{tag}.csv", index=False)
    pd.DataFrame(history).to_csv(WORK_DIR / f"epoch_history_{tag}.csv",
                                 index=False)

    result = {"experiment": spec["name"], "resize": resize, "arch": spec["arch"],
              "size": size, "balancing": spec["balancing"], "fold": fold_index,
              "best_epoch": best_epoch, "best_auc_epoch": best_auc_epoch,
              "checkpoint": str(WORK_DIR / f"{tag}.pth"),
              "val_labels": val_labels, "val_probs": val_probs,
              "val_groups": val_groups,
              "val": metrics_at(val_labels, val_probs, 0.5), "selection": best}
    del model, loaders, optimizer, scheduler, scaler, best_state
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()
    return result

# 3. Kết quả

In [15]:
log(f"Bắt đầu {RUN_MODE}: {len(EXPERIMENTS)} thí nghiệm × {len(FOLDS)} fold × "
    f"tối đa {EPOCHS} epoch")
_t0 = time.time()
RUNS = [run_fold(spec, fold)
        for spec in EXPERIMENTS
        for fold in FOLDS_TO_RUN]
log(f"\ntổng thời gian: {(time.time() - _t0) / 60:.1f} phút "
    f"({len(EXPERIMENTS)} thí nghiệm × {len(FOLDS_TO_RUN)} fold)")

Bắt đầu full: 1 thí nghiệm × 5 fold × tối đa 15 epoch

resnet18_v6  |  fold 0
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 187MB/s]


resnet18 | 224px | resize stretch | augment manh | balancing weighted | lr 1e-04 | epochs 15
chọn checkpoint: spec@sens97 → HSAS@97 → NLL → epoch sớm hơn
epoch  1/15  loss 0.1671  spec@sens97% 0.9342  HSAS 0.8409  AUC 0.9938  NLL 0.2858  first  <- best
epoch  2/15  loss 0.1078  spec@sens97% 0.9424  HSAS 0.8536  AUC 0.9928  NLL 0.1516  higher_specificity  <- best
epoch  3/15  loss 0.0857  spec@sens97% 0.9630  HSAS 0.8833  AUC 0.9954  NLL 0.1110  higher_specificity  <- best
epoch  4/15  loss 0.0735  spec@sens97% 0.9918  HSAS 0.9398  AUC 0.9975  NLL 0.0697  higher_specificity  <- best
epoch  5/15  loss 0.0678  spec@sens97% 0.9918  HSAS 0.9438  AUC 0.9981  NLL 0.1911  specificity_tie_higher_hsas  <- best
epoch  6/15  loss 0.0535  spec@sens97% 0.9671  HSAS 0.8891  AUC 0.9959  NLL 0.2611  lower_specificity
epoch  7/15  loss 0.0534  spec@sens97% 0.9588  HSAS 0.8742  AUC 0.9948  NLL 0.1418  lower_specificity
epoch  8/15  loss 0.0612  spec@sens97% 0.9753  HSAS 0.9152  AUC 0.9957  NLL 0.0819  lo

In [16]:
display(pd.DataFrame([
    {"fold": r["fold"], "epoch v6": r["best_epoch"],
     "epoch AUC cũ": r["best_auc_epoch"],
     "khác nhau": r["best_epoch"] != r["best_auc_epoch"],
     "lý do": r["selection"]["selection_reason"],
     "spec@sens97": round(r["selection"]["specificity"], 4),
     "HSAS@97": round(r["selection"]["hsas_97"], 4)}
    for r in RUNS]).set_index("fold"))

differing = sum(r["best_epoch"] != r["best_auc_epoch"] for r in RUNS)
print(f"\n{differing}/{len(RUNS)} fold chọn epoch khác quy tắc AUC cũ")
if differing == 0:
    print("  Quy tắc mới không đổi lựa chọn nào: ResNet18 vốn đã giữ đúng epoch,")
    print("  và C2 là một null control.")

,epoch v6,epoch AUC cũ,khác nhau,lý do,spec@sens97,HSAS@97
fold,,,,,,
0,5,10,True,specificity_tie_higher_hsas,0.9918,0.9438
1,14,12,True,specificity_hsas_tie_lower_nll,0.9959,0.9726
2,12,12,False,higher_specificity,0.9959,0.9767
3,4,4,False,higher_specificity,0.9672,0.9364
4,5,5,False,specificity_tie_higher_hsas,0.9959,0.9780



2/5 fold chọn epoch khác quy tắc AUC cũ


## 3.2. Ba ensemble, chọn hoàn toàn bằng OOF

E0 là mô hình đang khóa và nằm trong tập ứng viên. Nếu nó thắng, benchmark
không được mở.

In [17]:
ROOT_V4 = next((p for p in [LOCAL_PROJECT_ROOT / "notebooks/results_v4",
                            Path("/kaggle/input")] if p.is_dir()), None)
ROOT_V5 = next((p for p in [LOCAL_PROJECT_ROOT / "notebooks/results_v5",
                            Path("/kaggle/input")] if p.is_dir()), None)


def pooled_oof(root, name):
    """One out-of-fold row per group for a frozen model."""
    hits = sorted(root.rglob(f"predictions_oof_{name}_groups.csv"))
    if hits:
        return pd.read_csv(hits[0])[["group_id", "label", "p_pneumonia"]]
    parts = []
    for fold in range(5):
        path = sorted(root.rglob(
            f"validation_predictions_{name}_fold{fold}.csv"))[0]
        parts.append(pd.read_csv(path, usecols=["group_id", "class_id",
                                                "p_pneumonia"]))
    pooled = pd.concat(parts, ignore_index=True)
    return (pooled.groupby("group_id", as_index=False)
            .agg(label=("class_id", "first"),
                 p_pneumonia=("p_pneumonia", "mean")))


members = {
    "resnet_v4": pooled_oof(ROOT_V4, "stretch_manh"),
    "densenet_v5": pooled_oof(ROOT_V5, "densenet121_robust"),
}
own = pd.concat([pd.DataFrame({"group_id": r["val_groups"],
                               "class_id": r["val_labels"],
                               "p_pneumonia": r["val_probs"]}) for r in RUNS])
members["resnet_v6"] = (own.groupby("group_id", as_index=False)
                        .agg(label=("class_id", "first"),
                             p_pneumonia=("p_pneumonia", "mean")))

table = None
for name, frame in members.items():
    frame = frame.rename(columns={"p_pneumonia": name})
    table = frame if table is None else table.merge(frame,
                                                    on=["group_id", "label"])
table = table.set_index("group_id").sort_index()
y = table["label"].to_numpy()
print(f"{len(y):,} group chung cho cả ba mô hình")

CANDIDATES = {
    "E0 R_v4+D": {"resnet_v4": 0.50, "densenet_v5": 0.50},
    "E1 R_v6+D": {"resnet_v6": 0.50, "densenet_v5": 0.50},
    "E2 R_v4+R_v6+D": {"resnet_v4": 0.25, "resnet_v6": 0.25,
                       "densenet_v5": 0.50},
}
rows = []
for name, weights in CANDIDATES.items():
    score = sum(w * table[m].to_numpy() for m, w in weights.items())
    specificity, threshold = specificity_at_sensitivity(y, score)
    rows.append({"ensemble": name, "n_members": len(weights),
                 "specificity": specificity, "hsas_97": hsas_97(y, score),
                 "nll": group_nll(y, score), "threshold": threshold})
candidates = pd.DataFrame(rows)
display(candidates.round(4))

top = candidates.sort_values("specificity", ascending=False)
tied = top[(top["specificity"] - top.iloc[0]["specificity"]).abs()
           < SPECIFICITY_TIE]
if len(tied) > 1:
    tied = tied.sort_values("hsas_97", ascending=False)
    tied2 = tied[(tied["hsas_97"] - tied.iloc[0]["hsas_97"]).abs() < HSAS_TIE]
    if len(tied2) > 1:
        tied2 = tied2.sort_values(["nll", "n_members"])
    winner = tied2.iloc[0]
else:
    winner = tied.iloc[0]

print(f"\n=> CHỌN trên OOF: {winner['ensemble']}")
OPEN_BENCHMARK = not winner["ensemble"].startswith("E0")
print(f"   mở benchmark: {'CÓ' if OPEN_BENCHMARK else 'KHÔNG — E0 vẫn thắng, đóng băng mô hình cũ'}")
candidates.to_csv(WORK_DIR / "results_c2_ensembles.csv", index=False)

3,669 group chung cho cả ba mô hình


,ensemble,n_members,specificity,hsas_97,nll,threshold
0,E0 R_v4+D,2,0.9910,0.9594,0.0577,0.5873
1,E1 R_v6+D,2,0.9926,0.9450,0.0621,0.5692
2,E2 R_v4+R_v6+D,3,0.9918,0.9564,0.0581,0.5859



=> CHỌN trên OOF: E0 R_v4+D
   mở benchmark: KHÔNG — E0 vẫn thắng, đóng băng mô hình cũ


# 4. Benchmark — chỉ khi ensemble mới thắng OOF

Nếu E0 thắng thì ô dưới không chạy gì, và mô hình đang khóa được giữ nguyên.

In [18]:
if not OPEN_BENCHMARK:
    print("E0 thắng trên OOF. Benchmark KHÔNG được mở.")
    print("Mô hình kỹ thuật cuối cùng giữ nguyên: ResNet18 v4 + DenseNet121 v5.")
else:
    raise NotImplementedError(
        "Ensemble mới thắng OOF. Khóa thành viên và ngưỡng, băm lựa chọn, "
        "rồi chấm benchmark một lần trong một patch riêng.")

E0 thắng trên OOF. Benchmark KHÔNG được mở.
Mô hình kỹ thuật cuối cùng giữ nguyên: ResNet18 v4 + DenseNet121 v5.
